#### 연습
- data폴더 안에 가전 폴더의 모든 json파일을 하나의 데이터프레임으로 단순 행 결합
- 데이터에서 결측치를 확인
    - GeneralPolaruty 컬럼에서 결측치 발견
- 결측치가 포함된 데이터를 따로 저장 (na_df)
- 결측치를 제거
- 'RawText', 'GeneralPolarity' 컬럼을 제외한 나머지 컬럼 제외
- 'GeneralPolarity' 컬럼의 이름을 labels 변경
- RawText는 텍스트 정규화 (특수문자 제거, 2칸 이상의 공백 제외, 문자열 앞 뒤 공백 제거)
- labels 데이터에서 -1과 0은 0으로 1은 1로 데이터를 변경 -> 해당 컬럼의 dtype을 int변경 -> BERTmodel에서 선형 모델로 확률을 예측하기 때문에 labels가 위치 값
- train, test의 형태로 데이터를 9:1의 비율로 나눠준다.
    - labels를 기준으로 계층화 분할
- 데이터프레임을 Dataset의 형태로 변환
- tokens화 작업은 AutoTokenizer를 이용하여 모델의 이름은 skt/kobert-base-v1을 이용하여 토큰화
- 같은 모델을 로드하여 BertModel + Linear 모델 정의
- Trainer, TrainingArguments를 이용하여 학습
- 학습 -> na_df에서 상위 5개의 RawText를 확인하여 예측

In [167]:
import re
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, BertModel, Trainer, TrainingArguments

In [168]:
# 모든 가전 폴더의 데이터들의 단순 행 결합을 통한 데이터프레임 생성
df = pd.DataFrame()

# 폴더 안에 파일 불러와서 데이터프레임에 추가
for i in range(1, 5):
    if i == 1:
        for j in range(76, 89):
            data = pd.read_json(f"../data/가전/3-{i}.영상음향가전({j}).json")
            df = pd.concat([df, data], ignore_index=True)
    elif i == 2:
        for j in range(128, 141):
            data = pd.read_json(f"../data/가전/3-{i}.생활미용욕실가전({j}).json")
            df = pd.concat([df, data], ignore_index=True)
    elif i == 3:
        for j in range(127, 140):
            data = pd.read_json(f"../data/가전/3-{i}.주방가전({j}).json")
            df = pd.concat([df, data], ignore_index=True)
    else:
        for j in range(126, 129):
            data = pd.read_json(f"../data/가전/3-{i}.계절가전({j}).json")
            df = pd.concat([df, data], ignore_index=True)
            

len(df)


4056

In [169]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4056 entries, 0 to 4055
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   object 
 2   Source           4056 non-null   object 
 3   Domain           4056 non-null   object 
 4   MainCategory     4056 non-null   object 
 5   ProductName      4056 non-null   object 
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(6)
memory usage: 380.4+ KB


In [170]:
df['GeneralPolarity'].isna().sum()

np.int64(378)

In [171]:
# 결측치가 포함된 행을 따로 저장
df_na = df[df['GeneralPolarity'].isna()]
len(df_na)

378

In [172]:
df2 = df.dropna(subset=['GeneralPolarity']).reset_index(drop=True)
len(df2)

3678

In [173]:
df3 = df2[['RawText', 'GeneralPolarity']]

In [174]:
df3.rename(columns={'GeneralPolarity': 'labels'}, inplace=True)

C:\Users\abohv\AppData\Local\Temp\ipykernel_17128\4109285616.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3.rename(columns={'GeneralPolarity': 'labels'}, inplace=True)


In [175]:
df3

,RawText,labels
0,엄마가 갑자기 전화하시더니 집에서 사용하는 노래방 마이크가 사고 싶다고 하시네요.....,0.0
1,누가 사용하다가 반품한 것만 같은 제품이 와서 조금 당황스러웠어요ㅠㅠ일단 겉 부분에...,-1.0
2,노트북과 TV를 연결해서 모니터 하나에서 다 보려고 구매했어요.여러 제품과 비교하고...,0.0
3,너무 별로예요 ㅡㅡ… 이 정도 퀄리티인 줄 알았으면 안 샀을 거 같네요… 음질이 거...,-1.0
4,소음이 섞여서 나는 편이에요... 사이즈도 작고 휴대하기 간편해서 손이 자주 가긴 ...,-1.0
...,...,...
3673,사진으로는 잘 체감하지 못했는데 실제 실물을 보니 디자인이 조금 충격적이라고나 할까...,1.0
3674,이 에어컨의 제일 큰 장점은 조작법에 있는 것 같아요.인공지능 조작이 가능해서 한번...,1.0
3675,"이 제품을 추천하는 이유는요! 일단 LED를 통해 공기질을 눈으로 확인할 수 있고,...",1.0
3676,"인터넷에서 구매하였는데요, 이 공기청정기 처음 발견하고, 처음에는 오잉? 이게뭐지?...",1.0


In [176]:
df3['labels'].value_counts()

labels
 1.0    2220
 0.0     944
-1.0     514
Name: count, dtype: int64

In [177]:
# 텍스트 정규화 함수
def normalize_token_text(text:str) -> str:
    text= re.sub(r'[^가-힣a-zA-Z0-9\s\.]', " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df3['RawText'] = df3['RawText'].map(normalize_token_text) 

C:\Users\abohv\AppData\Local\Temp\ipykernel_17128\157300414.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['RawText'] = df3['RawText'].map(normalize_token_text)


In [178]:
for i in range(len(df3)):
    if df3.loc[i, 'labels'] == -1 or df3.loc[i, 'labels'] == 0:
        df3.loc[i, 'labels'] = 0
    else:
        df3.loc[i, 'labels'] = 1
        
df3['labels'] = df3['labels'].astype(int)

C:\Users\abohv\AppData\Local\Temp\ipykernel_17128\1930284321.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['labels'] = df3['labels'].astype(int)


In [179]:
train_df, test_df = train_test_split(df3, test_size=0.1, random_state=42, stratify=df3['labels'])

In [180]:
train_ds = Dataset.from_pandas(train_df)
test_ds= Dataset.from_pandas(test_df)

In [181]:
# 토큰화 -> AutoTokenizer를 이용하여 BERT 모델에서 사용하는 토큰화 작업을 로드
MODEL_NAME = "skt/kobert-base-v1"

# use_fast = Fa;se -> 기본(파이썬 기반) 토크나이저
    # KoBERT 모델은 sentencepiece 기반 토큰화
# use_fast = True -> 빠른 토크나이저 -> RUST, C 기반
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast = False)

def tok_fn(batch):
    # batch? -> 데이터의 묶음
    
    # truncation? -> 문장이 최대 입력 길이를 초과했을때 자동으로 지를것인가?
    # max_length -> 토큰의 최대 길이 -> 128 설정은 BERT 모델의 일반적인 설정
    result = tokenizer(batch['RawText'], truncation=True, max_length=128)
    # result -> input_ids, attention_mask, token_type_ids 등을 포함
    return result

In [182]:
train_tok = train_ds.map(tok_fn, batched=True, remove_columns=['RawText'])
test_tok  = test_ds.map(tok_fn, batched=False, remove_columns=['RawText'])

Map: 100%|██████████| 368/368 [00:00<00:00, 1791.02 examples/s]


In [193]:
class BERTClsHead(nn.Module):
    def __init__(self, model_name, num_label = 2, dropout = 0.1):
        # model_name -> 로드할 모델의 이름 
        # num_label -> 분류 클래스의 개수
        # dropout -> 데이터의 소실 비율 ( 과적합 방지 )

        # 설정 초기화
        super().__init__()

        # 사전에 학습된 BERT 모델을 로드 (백본)
        self.backbone = BertModel.from_pretrained(model_name)

        # BERT model에서의 output의 차원 개수 -> 768
        hidden = self.backbone.config.hidden_size

        # 과적합 방지를 위한 dropout 
        self.dropout = nn.Dropout(dropout)

        # 선형 모델  -> 2개의 class를 분류하는 모델 
        self.classifier = nn.Linear(hidden, num_label)

        # 패딩 토큰의 아이디 값을 백본 설정에 패딩 아이디에 대입 -> 확인차 대입 (안정성)
        self.backbone.config.pad_token_id = tokenizer.pad_token_id
        # 초기설정 완료
    
    # 순전파 함수 생성
    def forward(self, input_ids = None, 
                attention_mask = None, labels = None, **kwargs):
        # input_ids -> 토큰화 인코딩 처리가 완료된 문장
        # attention_mask -> 실제 단어 / 패딩 단어
        # labels -> 학습 시 정답의 라벨 ( 없으면 추론 모드 )

        # 백본에 입력 데이터를 대입 
        out = self.backbone(input_ids = input_ids, attention_mask = attention_mask)

        # [CLS] 토큰 벡터를 추출 
        # 입력의 첫번째 토큰 [CLS] -> 문장 전체를 대표하는 의미
        pooled = out.last_hidden_state[:, 0]

        # 일정 비율 데이터 소실
        drop_out_data = self.dropout(pooled)

        logits = self.classifier(drop_out_data)

        result = {'Logits' : logits}

        # labels의 데이터가 존재한다면 손실 계산
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            result['loss'] = loss
        
        return result

In [194]:
# 모델을 생성 
model = BERTClsHead(MODEL_NAME)

In [195]:
# 평가 함수 정의 (정확도, f1score)
def metrics(eval_pred):
    # eval_pred -> 예측값, 실젯값
    logits, y = eval_pred
    # logits [ x.xxx, x.xxx ]
    pred = logits.argmax(-1)
    return {
        'accuracy_score' : accuracy_score(y, pred), 
        'f1_score' : f1_score(y, pred)
    }

In [196]:
# TrainingArguments -> Trainer가 학습 할때 사용한 각종 설정 값들을 지정하는 객체 

args = TrainingArguments(
    # 학습된 모델의 결과들을 저장할 디렉토리 지정
    output_dir="./kobert_from_bertmodel", 
    # 배치의 크기를 설정 
    per_device_train_batch_size= 16,    # 학습시 cpu/gpu에 할당이 되는 배치의 크기  
    per_device_eval_batch_size= 16,     # 평가시 할당이 되는 배치의 크기 
    # 평가 및 저장 주기 설정 
    eval_strategy= "epoch",             # 한 epoch 마다 평가 수행
    save_strategy= 'epoch',             # 한 epoch 마다 모델을 저장
    # 학습 관련 설정 
    num_train_epochs= 2,                # 학습 epoch 수
    learning_rate= 5e-5,                # 옵티마이저의 학습율
    weight_decay= 0.01,                 # 가중치 감소 계수
    warmup_ratio= 0.1,                  # lr의 값을 올리는 비율
    logging_steps= 50,                  # 로그를 출력할 step의 간격
    # 모델 선택 및 저장 기준 
    load_best_model_at_end= True,       # 학습이 끝났을때 가장 성능이 좋은 모델을 자동 로드 
    metric_for_best_model= 'f1',        # 최고의 모델을 판단하는 검증 지표
    greater_is_better= True,            # 평가 지표가 높을 수록 좋은 모델인가?
    # 하드웨어 설정 
    fp16= torch.cuda.is_available(),    # cuda사용시 16-bit 혼합정밀도 학습을 할것인가
    use_mps_device= (
        torch.backends.mps.is_available() if not torch.cuda.is_available() else False
    ),                                  # Mac M1/M2 등 Apple silicon 가속기 사용 여부
    # 외부 로깅용 설정 
    report_to= []

)

In [197]:
# Trainer -> 모델의 학습을 자동으로 관리하는 Hugging Face의 고수준 API

trainer = Trainer(
    model = model,                  # 모델 선택
    args = args,                    # 학습에 사용한 설정값
    train_dataset= train_tok,       # 학습에 사용할 데이터
    eval_dataset= test_tok,         # 평가에 사용할 데이터
    tokenizer = tokenizer,          # 토큰화 함수
    compute_metrics= metrics        # 평가 시 사용할 검증 지표 함수
)

C:\Users\abohv\AppData\Local\Temp\ipykernel_17128\1394178951.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [198]:
# 평가 및 예측 테스트 

eval_res = trainer.evaluate()
print("평가의 결과 : ", eval_res)

c:\Users\abohv\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


평가의 결과 :  {'eval_loss': 0.6936671733856201, 'eval_model_preparation_time': 0.0027, 'eval_accuracy_score': 0.5108695652173914, 'eval_f1_score': 0.6356275303643725, 'eval_runtime': 41.5147, 'eval_samples_per_second': 8.864, 'eval_steps_per_second': 0.554}


In [199]:
samples = df_na['RawText'].tail(10)

In [200]:
samples = samples.to_list()

In [201]:
enc = tokenizer(
    samples,
    return_tensors = 'pt',       # pt -> pytorch 텐서형태로 변환
    padding = True,
    truncation = True
)

In [202]:
with torch.no_grad():
    out = model(**enc)
    probs = torch.softmax(
        out['Logits'], dim=-1
    ).cpu().numpy()

In [203]:
for s, p in zip(samples, probs):
    print(f"{s} : 부정 = {p[0]:.3f}, 긍정 = {p[1]:.3f} | 예측 = {p.argmax()})")

에어컨을 키면 너무 춥고, 끄자니 너무 더운 날씨에 차선으로 선택하게 된 에어쿨러!기대가 너무 커서 그 성능이 마음에 들지 않으면 어쩌지 했는데 이게 왠걸? 너무나 강력한 기능이 내 마음에 쏙 든다!마치 작은 에어컨처럼 여러 방향으로 회전이 가능해서 집안 곳곳에 시원한 바람을 전달해주고, 세부기능까지 지원되어 원하는 모드로 운전이 가능하다는 장점까지 정말 완벽하다.그런데 조작성이 조금 미흡해보이는 점이 상당히 아쉬움..요즘 전자제품은 보통 터치식으로 출시되는 것 같던데 이 제품은 아직도 버튼식을 고집하는 것 같다.아무래도 버튼식은 고장이 잘 날수도 있고 해서 가급적이면 터치식을 선호하는데 조작법이 매우 아쉬움.. : 부정 = 0.509, 긍정 = 0.491 | 예측 = 0)
가습기도 되고 선풍기도 되고 스탠드 조명도 되고 저 작은 사이즈에 기능이 다양하게 있어서 궁금해서 구매해봤습니다.사무실에서 책상에 올려 놓고 사용할 거라서 일단 작은 사이즈가 너무 마음에 들었고실용적이더라고요~~~ 그런데 가습량이 은근히 많아서 컴퓨터 옆에다가 두면 컴퓨터 고장납니다가습기 위에다가 프로펠러 꽂으면 바로 선풍기가 되는데 뭔가.. 물이 나오는 느낌입니닼 ㅋㅋ그래서 선풍기는 잘 사용 안 하게 되더라고요... 그리고 램프를 꽂으면 스탠드가 되는데 밝기가 밝아서 눈이 아픕니다.. 그래서 각도 조절이 꼭 필요합니다.. 디자인이 깜찍하고 예뻐서 마음에 드네요 : 부정 = 0.558, 긍정 = 0.442 | 예측 = 0)
집에서 사용하던 가습기가 고장이 나서 새로 구입하려 검색하다 이 제품을 구매했습니다~일단 거실에서 사용할 거라 용량이 컸으면 했는데 크기도 괜찮고 분사력도 좋은 편이네요.가습기가 스테인레스로 된 건 처음 사용해 보는데 더 위생적인 거 같고 좋아요~~가열식 스팀 가습기라 세균 걱정없이 기존의 가습기보다 따뜻하게 사용할 수 있을 거 같네요.이 점이 마음에 들어서 구매를 하게 된 건데 실제로 받아보니 더 만족스럽습니다.별 하나를 뺀 이유는 소음이 적다고 해서 샀는데 생각보다 

In [204]:
df_na['RawText'].tail(10)

3825    에어컨을 키면 너무 춥고, 끄자니 너무 더운 날씨에 차선으로 선택하게 된 에어쿨러!...
3849    가습기도 되고 선풍기도 되고 스탠드 조명도 되고 저 작은 사이즈에 기능이 다양하게 ...
3854    집에서 사용하던 가습기가 고장이 나서 새로 구입하려 검색하다 이 제품을 구매했습니다...
3889    화장실에 놓고 쓰려고 주문 했어요. 생활 방수 기능이 있다고 해서 화장실에 놓고 쓰...
3893    OOO에서 난방기가 있다고 해서 디자인은 당연 믿고 샀구요. 전원을 켠 후 바로 따...
3897    대용량이고 세척이 간편하다는 얘기에 구매를 했는데 저는 별로인 거 같아요...생각했...
3900    요거 진짜 진짜 물건입니다. 처음엔 디자인이 너무 귀여워서 주문하게 되었는데요. 작...
3924    지인의 추천으로 믿고 바로 구매를 해서 현재도 사용중입니다좀 더 많은 분들에게 도움...
3932    다른 에어쿨러와 다르게 슬림한 디자인이 마음에 들어요.심플한 디자인 덕분에 집안 어...
3944    이번에 어머님 댁에 공기청정기가 필요하다고 하셔서 하나 장만해드렸어요. 전체적으로 ...
Name: RawText, dtype: object